# Solutions to project: the Game of Life

A break from particles: [Conway's Game of Life](https://en.wikipedia.org/wiki/Conway%27s_Game_of_Life), the famous cellular automaton, with the same NumPy tools as the Higgs project.
The world is a grid of cells, each dead (0) or alive (1), and one rule decides the next step:

- a living cell survives if it has 2 or 3 living neighbours
- a dead cell comes to life if it has exactly 3 living neighbours

Each of the 8 neighbours counts, and the world wraps around at the edges.
You will first watch a slow pure-Python version, then replace it with arrays.

*(Adapted from Henry Schreiner's HSF-India tutorial.)*

**How to work.** Replace every `...` with your own code. When a part is done, or you are properly stuck after five minutes, compare with the matching `solutions-` notebook.

In [ ]:
import numpy as np

`new_world` places a pattern called a *puffer* in an otherwise empty 32 × 128 world.
`show` prints the world as text: `@` is alive, `.` is dead.
Make the notebook wide enough to see the whole box.

In [ ]:
HEIGHT = 32
WIDTH = 128

def new_world():
    world = np.zeros((HEIGHT, WIDTH), dtype=np.int32)
    for y, x in [
        ( 4, 125), ( 3, 124), ( 3, 123), ( 3, 122), ( 3, 121), ( 3, 120), ( 3, 119),
        ( 4, 119), ( 5, 119), ( 6, 120), ( 9, 121), ( 9, 120), (10, 121), (10, 119),
        (11, 121), (12, 125), (13, 124), (13, 123), (13, 122), (13, 121), (13, 120),
        (13, 119), (14, 119), (15, 119), (16, 120),
    ]:
        world[y, x] = 1
    return world


def show(world):
    for row in world:
        print("|" + "".join("@" if cell else "." for cell in row) + "|")


world = new_world()
show(world)

## Part 1: Counting

The world is one array, so NumPy answers questions about all of it at once.

**Task.** How many cells are alive? A sum over 0s and 1s is a count.

In [ ]:
n_alive = world.sum()

n_alive

This pure-Python step follows the rule cell by cell.
The `%` makes the indices wrap around at the edges.
It works, and it is the reference your array version must match.

In [ ]:
def step_python(world):
    new = np.zeros_like(world)
    for y in range(HEIGHT):
        for x in range(WIDTH):
            neighbours = 0
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    if (dy, dx) != (0, 0):
                        neighbours += world[(y + dy) % HEIGHT, (x + dx) % WIDTH]
            if world[y, x] == 1 and neighbours in (2, 3):
                new[y, x] = 1
            if world[y, x] == 0 and neighbours == 3:
                new[y, x] = 1
    return new


show(step_python(world))

## Part 2: Counting neighbours with arrays

`np.roll` shifts a whole array, with wrap-around built in.

**Tools.** `np.roll(world, 1, axis=0)` is the world shifted down one row, so each cell of it holds its *upper* neighbour. `np.roll(np.roll(world, 1, axis=0), 1, axis=1)` shifts diagonally.

**Task.** Build `neighbours`: an array in which every cell holds its number of living neighbours. Add one shifted world per neighbour, all 8 of them.

In [ ]:
def count_neighbours(world):
    total = np.zeros_like(world)
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if (dy, dx) != (0, 0):
                total += np.roll(np.roll(world, dy, axis=0), dx, axis=1)
    return total


neighbours = count_neighbours(world)
neighbours

## Part 3: The rule, as masks

The two halves of the rule are two masks.

**Task.** Complete `step_numpy`: survivors are living cells with 2 or 3 neighbours, births are dead cells with exactly 3. The new world is `1` where either mask is `True`.
Remember `&`, `|` and parentheses from the lecture.

In [ ]:
def step_numpy(world):
    neighbours = count_neighbours(world)
    survive = (world == 1) & ((neighbours == 2) | (neighbours == 3))
    born = (world == 0) & (neighbours == 3)
    return (survive | born).astype(np.int32)


step_numpy(world)

Your array version and the loop version must agree everywhere:

In [ ]:
np.array_equal(step_numpy(world), step_python(world))

## Part 4: Run it

**Task.** Apply `step_numpy` 100 times, starting from a fresh world, and show the result.

In [ ]:
world = new_world()
for _ in range(100):
    world = step_numpy(world)


show(world)
world

## Bonus: How much faster?

`%timeit` runs a cell many times and reports the average.

In [ ]:
%timeit step_python(world)

In [ ]:
%timeit step_numpy(world)

**Done.** The same moves as the Higgs project: whole-array arithmetic instead of loops, masks instead of `if`, and a sum to count. The puffer, by the way, flies left forever, leaving smoke behind.